# Model Experiment: Temporal Fusion Transformer

TFT-inspired neural forecasting model for weekly Walmart Store-Dept sales.

The goal is to test whether sequence model that uses static metadata, past sales history and known future covariates can improve over simpler neural baselines such as DLinear.

We use point forecasting instead of quantile forecasting because the Kaggle metric is Weighted Mean Absolute Error (WMAE), not probabilistic forecasting metric.

Temporal Fusion Transformer is designed for multi-horizon time-series forecasting with several feature types:

- static features: Store, Dept, Type, Size
- past target history: previous Weekly_Sales values
- past covariates: historical calendar/economic features
- known future covariates: holiday flags, week seasonality, and other features known for the forecast horizon

For our task:

- input context: previous 52 weeks
- prediction horizon: next 39 weeks
- output: 39 future Weekly_Sales predictions for each Store-Dept series

Unlike DLinear which uses mostly linear mappings from past target values, TFT learns nonlinear sequence representations using static embeddings, recurrent layers, attention, gating and future covariates.

Main experiment: TFT with stable covariates

The initial TFT model uses:

- past target sequence
- static categorical embeddings for Store, Dept, and Type
- static real feature Size_scaled
- stable known covariates such as IsHoliday, holiday flags, Week_sin/Week_cos, Temperature, Fuel_Price, CPI and Unemployment

Based on the DLinear-X experiments, raw MarkDown features are excluded from the first TFT version because they made neural covariate models less stable and were problematic for the calendar-aligned validation split.

Validation strategy:

- last-39 validation is used for the initial TFT grid search because it provides many more training windows
- calendar-aligned validation is used as a realism check because it was closer to Kaggle performance for DLinear

## Setup and Imports

In [2]:
from pathlib import Path
import sys
import os
import json
import random
from copy import deepcopy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [3]:
# cwd = Path.cwd().resolve()
# repo_root = cwd if (cwd / "src").exists() else cwd.parent

# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# ==========================================

# repo_root = Path("/content/drive/MyDrive/Machine Learning/Walmart").resolve()

# assert (repo_root / "src").exists(), f"src not found at {repo_root / 'src'}"

# print("Repo root:", repo_root)
# print("src exists:", (repo_root / "src").exists())
# print("data exists:", (repo_root / "data" / "raw").exists())

# os.chdir(repo_root)

# ==========================================

import shutil
import zipfile

KAGGLE_INPUT = Path("/kaggle/input")
repo_root = Path("/kaggle/working/Walmart").resolve()
data_raw_dir = repo_root / "data" / "raw"

repo_root.mkdir(parents=True, exist_ok=True)
data_raw_dir.mkdir(parents=True, exist_ok=True)

print("Available /kaggle/input folders:")
for p in KAGGLE_INPUT.iterdir():
    print(" -", p)


src_candidates = [
    p for p in KAGGLE_INPUT.rglob("src")
    if p.is_dir() and (p / "data").exists() and (p / "features").exists()
]

if not src_candidates:
    raise FileNotFoundError(
        "Could not find your project src/ folder under /kaggle/input. "
        "Make sure your Kaggle Dataset contains the src directory."
    )

source_src = src_candidates[0]
target_src = repo_root / "src"

if target_src.exists():
    shutil.rmtree(target_src)

shutil.copytree(source_src, target_src)

print("\nCopied src from:", source_src)
print("Copied src to:", target_src)

required_csvs = ["train.csv", "test.csv", "features.csv", "stores.csv"]

for csv_name in required_csvs:
    matches = list(KAGGLE_INPUT.rglob(csv_name))
    if matches:
        shutil.copy2(matches[0], data_raw_dir / csv_name)
        print(f"Copied {csv_name} from:", matches[0])

for zip_path in KAGGLE_INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()
            wanted = [name for name in names if Path(name).name in required_csvs]

            for name in wanted:
                out_name = Path(name).name
                with z.open(name) as src_file, open(data_raw_dir / out_name, "wb") as dst_file:
                    shutil.copyfileobj(src_file, dst_file)

                print(f"Extracted {out_name} from:", zip_path)
    except zipfile.BadZipFile:
        pass

missing = [name for name in required_csvs if not (data_raw_dir / name).exists()]

if missing:
    print("\nFiles currently in data/raw:")
    for p in data_raw_dir.iterdir():
        print(" -", p.name)

    raise FileNotFoundError(f"Missing required raw files: {missing}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

print("\nRepo root:", repo_root)
print("src exists:", (repo_root / "src").exists())
print("data exists:", data_raw_dir.exists())
print("Raw data files:", sorted(p.name for p in data_raw_dir.iterdir()))

from src.data import load_raw_data, last_n_weeks_split, calendar_aligned_split
from src.features import WalmartBasePreprocessor, WalmartNeuralPreprocessor
from src.datasets import (
    WalmartPrecomputedTrainingWindowDataset,
    WalmartPrecomputedForecastWindowDataset,
    FastTensorDataLoader,
)

Available /kaggle/input folders:
 - /kaggle/input/competitions
 - /kaggle/input/datasets

Copied src from: /kaggle/input/datasets/myvari/walmart-project-code/src
Copied src to: /kaggle/working/Walmart/src
Copied stores.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
Extracted train.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
Extracted features.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
Extracted test.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip

Repo root: /kaggle/working/Walmart
src exists: True
data exists: True
Raw data files: ['features.csv', 'stores.csv', 'test.csv', 'train.csv']


In [4]:
DATA_DIR = repo_root / "data" / "raw"

CONTEXT_LENGTH = 52
PREDICTION_LENGTH = 39

BATCH_SIZE = 256
SEED = 42

MLFLOW_EXPERIMENT_NAME = "TFT_Training"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Device:", DEVICE)

Repo root: /kaggle/working/Walmart
Data dir: /kaggle/working/Walmart/data/raw
Device: cuda


In [5]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Dagshub/Mlflow initialization

In [6]:
pip install dagshub mlflow pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import dagshub
import mlflow

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0ef0c99d-c214-4793-8c8e-dbe1826439df&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=75fe56a69397abc53471bfa7960a2b4ced27baae7a7736f2db066acff6dc3949




Accessing as myvari

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## Load Dataset and Time Split

In [8]:
train, test, stores, features = load_raw_data(DATA_DIR)

for df in [train, test, features]:
    df["Date"] = pd.to_datetime(df["Date"])

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("features:", features.shape)

print("\nTrain date range:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date range:")
print(test["Date"].min(), "->", test["Date"].max())

display(train.head())
display(test.head())
display(stores.head())
display(features.head())

train: (421570, 5)
test: (115064, 4)
stores: (45, 3)
features: (8190, 12)

Train date range:
2010-02-05 00:00:00 -> 2012-10-26 00:00:00

Test date range:
2012-11-02 00:00:00 -> 2013-07-26 00:00:00


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [9]:
assert {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}.issubset(train.columns)
assert {"Store", "Dept", "Date", "IsHoliday"}.issubset(test.columns)
assert {"Store", "Type", "Size"}.issubset(stores.columns)
assert {"Store", "Date", "IsHoliday"}.issubset(features.columns)

print("Raw data checks passed.")

Raw data checks passed.


In [10]:
# split A: last 39 weeks of train

train_raw_part, valid_raw_part = last_n_weeks_split(
    train,
    n_weeks=PREDICTION_LENGTH,
    date_col="Date",
)

print("Last-39 split")
print("train_raw_part:", train_raw_part.shape)
print("valid_raw_part:", valid_raw_part.shape)

print("\nTrain split date range:")
print(train_raw_part["Date"].min(), "->", train_raw_part["Date"].max())

print("\nValidation split date range:")
print(valid_raw_part["Date"].min(), "->", valid_raw_part["Date"].max())

print("\nUnique train dates:", train_raw_part["Date"].nunique())
print("Unique validation dates:", valid_raw_part["Date"].nunique())

assert train_raw_part["Date"].max() < valid_raw_part["Date"].min()
assert valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Last-39 split checks passed.")

Last-39 split
train_raw_part: (305982, 5)
valid_raw_part: (115588, 5)

Train split date range:
2010-02-05 00:00:00 -> 2012-01-27 00:00:00

Validation split date range:
2012-02-03 00:00:00 -> 2012-10-26 00:00:00

Unique train dates: 104
Unique validation dates: 39
Last-39 split checks passed.


In [11]:
# split B: calendar-aligned validation

calendar_train_raw_part, calendar_valid_raw_part = calendar_aligned_split(
    train,
    valid_start="2011-11-04",
    valid_end="2012-07-27",
    date_col="Date",
)

print("Calendar-aligned split")
print("calendar_train_raw_part:", calendar_train_raw_part.shape)
print("calendar_valid_raw_part:", calendar_valid_raw_part.shape)

print("\nCalendar train date range:")
print(calendar_train_raw_part["Date"].min(), "->", calendar_train_raw_part["Date"].max())

print("\nCalendar validation date range:")
print(calendar_valid_raw_part["Date"].min(), "->", calendar_valid_raw_part["Date"].max())

print("\nUnique calendar train dates:", calendar_train_raw_part["Date"].nunique())
print("Unique calendar validation dates:", calendar_valid_raw_part["Date"].nunique())

assert calendar_train_raw_part["Date"].max() < calendar_valid_raw_part["Date"].min()
assert calendar_valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Calendar-aligned split checks passed.")

Calendar-aligned split
calendar_train_raw_part: (267184, 5)
calendar_valid_raw_part: (115856, 5)

Calendar train date range:
2010-02-05 00:00:00 -> 2011-10-28 00:00:00

Calendar validation date range:
2011-11-04 00:00:00 -> 2012-07-27 00:00:00

Unique calendar train dates: 91
Unique calendar validation dates: 39
Calendar-aligned split checks passed.


## Base + neural preprocessing

In [12]:
base_preprocessor = WalmartBasePreprocessor()
base_preprocessor.fit(stores, features)

# last-39
last39_train_base = base_preprocessor.transform(train_raw_part)
last39_valid_base = base_preprocessor.transform(valid_raw_part)

# calendar-aligned
calendar_train_base = base_preprocessor.transform(calendar_train_raw_part)
calendar_valid_base = base_preprocessor.transform(calendar_valid_raw_part)

print("Last-39 base:")
print("last39_train_base:", last39_train_base.shape)
print("last39_valid_base:", last39_valid_base.shape)

print("\nCalendar base:")
print("calendar_train_base:", calendar_train_base.shape)
print("calendar_valid_base:", calendar_valid_base.shape)

assert len(last39_train_base) == len(train_raw_part)
assert len(last39_valid_base) == len(valid_raw_part)

assert len(calendar_train_base) == len(calendar_train_raw_part)
assert len(calendar_valid_base) == len(calendar_valid_raw_part)

display(last39_train_base.head())

Last-39 base:
last39_train_base: (305982, 42)
last39_valid_base: (115588, 42)

Calendar base:
calendar_train_base: (267184, 42)
calendar_valid_base: (115856, 42)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,...,markdown_available_period,Year,Month,WeekOfYear,Week_sin,Week_cos,IsSuperBowl,IsLaborDay,IsThanksgiving,IsChristmas
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,...,0,2010,2,5,0.568065,0.822984,0,0,0,0
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,...,0,2010,2,6,0.663123,0.748511,1,0,0,0
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,...,0,2010,2,7,0.748511,0.663123,0,0,0,0
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,...,0,2010,2,8,0.822984,0.568065,0,0,0,0
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,...,0,2010,3,9,0.885456,0.464723,0,0,0,0


In [13]:
required_base_cols = [
    "Store", "Dept", "Date", "Weekly_Sales",
    "Type", "Size",
    "IsHoliday",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "MarkDown1_was_missing", "MarkDown2_was_missing", "MarkDown3_was_missing",
    "MarkDown4_was_missing", "MarkDown5_was_missing",
    "total_markdown", "abs_total_markdown",
    "positive_markdown_sum", "negative_markdown_sum",
    "markdown_missing_count", "markdown_available_period",
    "has_markdown_signal",
    "Week_sin", "Week_cos",
    "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
]

base_panels = {
    "last39_train_base": last39_train_base,
    "last39_valid_base": last39_valid_base,
    "calendar_train_base": calendar_train_base,
    "calendar_valid_base": calendar_valid_base,
}

for name, df in base_panels.items():
    missing = [c for c in required_base_cols if c not in df.columns]
    print(name, "missing:", missing)
    assert not missing, f"{name} is missing columns: {missing}"

print("Base preprocessing checks passed.")

last39_train_base missing: []
last39_valid_base missing: []
calendar_train_base missing: []
calendar_valid_base missing: []
Base preprocessing checks passed.


In [14]:
# last-39 neural preprocessing
last39_neural_preprocessor = WalmartNeuralPreprocessor()
last39_neural_preprocessor.fit(last39_train_base)

last39_train_panel = last39_neural_preprocessor.transform(last39_train_base)
last39_valid_panel = last39_neural_preprocessor.transform(last39_valid_base)

last39_dataset_cols = last39_neural_preprocessor.get_dataset_columns()

# calendar-aligned neural preprocessing
calendar_neural_preprocessor = WalmartNeuralPreprocessor()
calendar_neural_preprocessor.fit(calendar_train_base)

calendar_train_panel = calendar_neural_preprocessor.transform(calendar_train_base)
calendar_valid_panel = calendar_neural_preprocessor.transform(calendar_valid_base)

calendar_dataset_cols = calendar_neural_preprocessor.get_dataset_columns()

print("Last-39 panels:")
print("last39_train_panel:", last39_train_panel.shape)
print("last39_valid_panel:", last39_valid_panel.shape)

print("\nCalendar panels:")
print("calendar_train_panel:", calendar_train_panel.shape)
print("calendar_valid_panel:", calendar_valid_panel.shape)

print("\nDataset columns:")
for key, value in calendar_dataset_cols.items():
    print(f"{key}: {value}")

Last-39 panels:
last39_train_panel: (305982, 66)
last39_valid_panel: (115588, 66)

Calendar panels:
calendar_train_panel: (267184, 66)
calendar_valid_panel: (115856, 66)

Dataset columns:
target_col: Weekly_Sales_scaled
series_col: series_id
static_cat_cols: ['Store_id', 'Dept_id', 'Type_id']
static_real_cols: ['Size_scaled']
known_future_real_cols: ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


In [15]:
expected_neural_cols = [
    "series_id",
    "Store_id", "Dept_id", "Type_id",
    "Weekly_Sales_scaled",
    "target_mean", "target_std",
]

neural_panels = {
    "last39_train_panel": last39_train_panel,
    "last39_valid_panel": last39_valid_panel,
    "calendar_train_panel": calendar_train_panel,
    "calendar_valid_panel": calendar_valid_panel,
}

for name, df in neural_panels.items():
    for col in expected_neural_cols:
        assert col in df.columns, f"Missing from {name}: {col}"

    assert df["Weekly_Sales_scaled"].notna().all(), f"NaN target scale in {name}"
    assert df["target_mean"].notna().all(), f"NaN target_mean in {name}"
    assert df["target_std"].notna().all(), f"NaN target_std in {name}"
    assert (df["target_std"] > 0).all(), f"Non-positive target_std in {name}"

print("Neural preprocessing checks passed.")

Neural preprocessing checks passed.


In [16]:
print("Static categorical columns:")
print(calendar_dataset_cols["static_cat_cols"])

print("\nStatic real columns:")
print(calendar_dataset_cols["static_real_cols"])

print("\nKnown future/time-varying feature columns:")
print("Count:", len(calendar_dataset_cols["known_future_real_cols"]))
print(calendar_dataset_cols["known_future_real_cols"])

print("\nLast-39 feature count:", len(last39_dataset_cols["known_future_real_cols"]))
print("Calendar feature count:", len(calendar_dataset_cols["known_future_real_cols"]))

assert last39_dataset_cols["known_future_real_cols"] == calendar_dataset_cols["known_future_real_cols"]
assert last39_dataset_cols["static_cat_cols"] == calendar_dataset_cols["static_cat_cols"]
assert last39_dataset_cols["static_real_cols"] == calendar_dataset_cols["static_real_cols"]

print("Feature column definitions match across splits.")

Static categorical columns:
['Store_id', 'Dept_id', 'Type_id']

Static real columns:
['Size_scaled']

Known future/time-varying feature columns:
Count: 28
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']

Last-39 feature count: 28
Calendar feature count: 28
Feature column definitions match across splits.


##  Dataset and DataLoaders

In [17]:
def remove_markdown_features(cols: list[str]) -> list[str]:
    return [
        col for col in cols
        if "markdown" not in col.lower()
    ]


# primary TFT feature set - stable covariates only
tft_calendar_known_future_cols = remove_markdown_features(
    calendar_dataset_cols["known_future_real_cols"]
)

tft_last39_known_future_cols = remove_markdown_features(
    last39_dataset_cols["known_future_real_cols"]
)

tft_calendar_cols = {
    "target_col": calendar_dataset_cols["target_col"],
    "series_col": calendar_dataset_cols["series_col"],
    "static_cat_cols": calendar_dataset_cols["static_cat_cols"],
    "static_real_cols": calendar_dataset_cols["static_real_cols"],
    "known_future_real_cols": tft_calendar_known_future_cols,
}

tft_last39_cols = {
    "target_col": last39_dataset_cols["target_col"],
    "series_col": last39_dataset_cols["series_col"],
    "static_cat_cols": last39_dataset_cols["static_cat_cols"],
    "static_real_cols": last39_dataset_cols["static_real_cols"],
    "known_future_real_cols": tft_last39_known_future_cols,
}

print("TFT calendar known future features:", len(tft_calendar_cols["known_future_real_cols"]))
print(tft_calendar_cols["known_future_real_cols"])

print("\nTFT last-39 known future features:", len(tft_last39_cols["known_future_real_cols"]))
print(tft_last39_cols["known_future_real_cols"])

removed_cols = [
    col for col in calendar_dataset_cols["known_future_real_cols"]
    if col not in tft_calendar_cols["known_future_real_cols"]
]

print("\nRemoved MarkDown-related features:")
for col in removed_cols:
    print("-", col)

TFT calendar known future features: 11
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas']

TFT last-39 known future features: 11
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas']

Removed MarkDown-related features:
- MarkDown1_scaled
- MarkDown2_scaled
- MarkDown3_scaled
- MarkDown4_scaled
- MarkDown5_scaled
- total_markdown_scaled
- abs_total_markdown_scaled
- positive_markdown_sum_scaled
- negative_markdown_sum_scaled
- markdown_missing_count_scaled
- has_markdown_signal
- markdown_available_period
- MarkDown1_was_missing
- MarkDown2_was_missing
- MarkDown3_was_missing
- MarkDown4_was_missing
- MarkDown5_was_missing


In [18]:
def make_full_horizon_validation_panel(
    valid_panel: pd.DataFrame,
    prediction_length: int,
    series_col: str = "series_id",
) -> tuple[pd.DataFrame, pd.Index]:
    group_sizes = valid_panel.groupby(series_col).size()

    full_horizon_series = group_sizes[
        group_sizes == prediction_length
    ].index

    valid_panel_full = valid_panel[
        valid_panel[series_col].isin(full_horizon_series)
    ].copy()

    assert len(valid_panel_full) == len(full_horizon_series) * prediction_length

    return valid_panel_full, full_horizon_series

In [19]:
calendar_valid_panel_full, calendar_full_horizon_series = make_full_horizon_validation_panel(
    calendar_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=tft_calendar_cols["series_col"],
)

last39_valid_panel_full, last39_full_horizon_series = make_full_horizon_validation_panel(
    last39_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=tft_last39_cols["series_col"],
)

print("Calendar validation:")
print("total series:", calendar_valid_panel["series_id"].nunique())
print("full-horizon series:", len(calendar_full_horizon_series))
print("calendar_valid_panel rows:", len(calendar_valid_panel))
print("calendar_valid_panel_full rows:", len(calendar_valid_panel_full))

print("\nLast-39 validation:")
print("total series:", last39_valid_panel["series_id"].nunique())
print("full-horizon series:", len(last39_full_horizon_series))
print("last39_valid_panel rows:", len(last39_valid_panel))
print("last39_valid_panel_full rows:", len(last39_valid_panel_full))

Calendar validation:
total series: 3233
full-horizon series: 2762
calendar_valid_panel rows: 115856
calendar_valid_panel_full rows: 107718

Last-39 validation:
total series: 3204
full-horizon series: 2762
last39_valid_panel rows: 115588
last39_valid_panel_full rows: 107718


In [20]:
def build_tft_data_bundle(
    train_panel: pd.DataFrame,
    valid_panel_full: pd.DataFrame,
    cols: dict,
    batch_size: int = BATCH_SIZE,
) -> dict:
    train_dataset = WalmartPrecomputedTrainingWindowDataset(
        train_panel,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    valid_dataset = WalmartPrecomputedForecastWindowDataset(
        history_df=train_panel,
        future_df=valid_panel_full,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    train_loader = FastTensorDataLoader(
        train_dataset.tensors,
        batch_size=batch_size,
        shuffle=True,
    )

    valid_loader = FastTensorDataLoader(
        valid_dataset.tensors,
        batch_size=batch_size,
        shuffle=False,
    )

    static_cat_cardinalities = [
        int(max(train_panel[col].max(), valid_panel_full[col].max())) + 1
        for col in cols["static_cat_cols"]
    ]

    num_static_reals = len(cols["static_real_cols"])
    num_known_reals = len(cols["known_future_real_cols"])
    holiday_feature_idx = cols["known_future_real_cols"].index("IsHoliday")

    assert train_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
    assert train_dataset.tensors["future_target"].shape[1] == PREDICTION_LENGTH

    assert train_dataset.tensors["past_known_reals"].shape[2] == num_known_reals
    assert train_dataset.tensors["future_known_reals"].shape[2] == num_known_reals
    assert valid_dataset.tensors["past_known_reals"].shape[2] == num_known_reals
    assert valid_dataset.tensors["future_known_reals"].shape[2] == num_known_reals

    assert train_dataset.tensors["static_categoricals"].shape[1] == len(cols["static_cat_cols"])
    assert valid_dataset.tensors["static_categoricals"].shape[1] == len(cols["static_cat_cols"])

    assert train_dataset.tensors["static_reals"].shape[1] == num_static_reals
    assert valid_dataset.tensors["static_reals"].shape[1] == num_static_reals

    return {
        "cols": cols,
        "train_dataset": train_dataset,
        "valid_dataset": valid_dataset,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "static_cat_cardinalities": static_cat_cardinalities,
        "num_static_reals": num_static_reals,
        "num_known_reals": num_known_reals,
        "holiday_feature_idx": holiday_feature_idx,
        "batch_size": batch_size,
    }

In [21]:
tft_calendar_data = build_tft_data_bundle(
    train_panel=calendar_train_panel,
    valid_panel_full=calendar_valid_panel_full,
    cols=tft_calendar_cols,
    batch_size=BATCH_SIZE,
)

tft_last39_data = build_tft_data_bundle(
    train_panel=last39_train_panel,
    valid_panel_full=last39_valid_panel_full,
    cols=tft_last39_cols,
    batch_size=BATCH_SIZE,
)

print("Calendar TFT data:")
print("train windows:", len(tft_calendar_data["train_dataset"]))
print("valid forecast series:", len(tft_calendar_data["valid_dataset"]))
print("train batches:", len(tft_calendar_data["train_loader"]))
print("valid batches:", len(tft_calendar_data["valid_loader"]))
print("static cat cardinalities:", tft_calendar_data["static_cat_cardinalities"])
print("num known reals:", tft_calendar_data["num_known_reals"])
print("num static reals:", tft_calendar_data["num_static_reals"])
print("holiday idx:", tft_calendar_data["holiday_feature_idx"])

print("\nLast-39 TFT data:")
print("train windows:", len(tft_last39_data["train_dataset"]))
print("valid forecast series:", len(tft_last39_data["valid_dataset"]))
print("train batches:", len(tft_last39_data["train_loader"]))
print("valid batches:", len(tft_last39_data["valid_loader"]))
print("static cat cardinalities:", tft_last39_data["static_cat_cardinalities"])
print("num known reals:", tft_last39_data["num_known_reals"])
print("num static reals:", tft_last39_data["num_static_reals"])
print("holiday idx:", tft_last39_data["holiday_feature_idx"])

Calendar TFT data:
train windows: 2683
valid forecast series: 2762
train batches: 11
valid batches: 11
static cat cardinalities: [46, 82, 4]
num known reals: 11
num static reals: 1
holiday idx: 6

Last-39 TFT data:
train windows: 38464
valid forecast series: 2762
train batches: 151
valid batches: 11
static cat cardinalities: [46, 82, 4]
num known reals: 11
num static reals: 1
holiday idx: 6


In [22]:
calendar_batch = next(iter(tft_calendar_data["train_loader"]))

print("Calendar train batch:")
for key, value in calendar_batch.items():
    print(key, tuple(value.shape), value.dtype)

valid_calendar_batch = next(iter(tft_calendar_data["valid_loader"]))

print("\nCalendar validation batch:")
for key, value in valid_calendar_batch.items():
    print(key, tuple(value.shape), value.dtype)

Calendar train batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 11) torch.float32
future_known_reals (256, 39, 11) torch.float32
static_categoricals (256, 3) torch.int64
static_reals (256, 1) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32

Calendar validation batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 11) torch.float32
future_known_reals (256, 39, 11) torch.float32
static_categoricals (256, 3) torch.int64
static_reals (256, 1) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32
store (256,) torch.int64
dept (256,) torch.int64


In [23]:
import time

def time_loader(loader, name="loader", max_batches=None):
    t0 = time.perf_counter()
    n_batches = 0
    n_samples = 0

    for batch in loader:
        n_batches += 1
        n_samples += batch["past_target"].shape[0]

        if max_batches is not None and n_batches >= max_batches:
            break

    elapsed = time.perf_counter() - t0

    print(f"{name}: {elapsed:.2f}s, batches={n_batches}, samples={n_samples}")
    print(f"seconds/batch: {elapsed / max(n_batches, 1):.4f}")


time_loader(tft_calendar_data["train_loader"], "TFT calendar train loader")
time_loader(tft_calendar_data["valid_loader"], "TFT calendar valid loader")
time_loader(tft_last39_data["train_loader"], "TFT last-39 train loader")
time_loader(tft_last39_data["valid_loader"], "TFT last-39 valid loader")

TFT calendar train loader: 0.01s, batches=11, samples=2683
seconds/batch: 0.0005
TFT calendar valid loader: 0.00s, batches=11, samples=2762
seconds/batch: 0.0004
TFT last-39 train loader: 0.05s, batches=151, samples=38464
seconds/batch: 0.0004
TFT last-39 valid loader: 0.00s, batches=11, samples=2762
seconds/batch: 0.0003


In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="TFT_Preprocessing") as run:
    mlflow.log_param("model_family", "TFT")
    mlflow.log_param("context_length", CONTEXT_LENGTH)
    mlflow.log_param("prediction_length", PREDICTION_LENGTH)
    mlflow.log_param("base_preprocessor", "WalmartBasePreprocessor")
    mlflow.log_param("neural_preprocessor", "WalmartNeuralPreprocessor")
    mlflow.log_param("target_scaling", "series_mean_std_with_dept_store_global_fallback")
    mlflow.log_param("primary_validation_strategy", "calendar_aligned_39_weeks")
    mlflow.log_param("secondary_validation_strategy", "last_39_weeks")
    mlflow.log_param("feature_set", "stable_no_markdowns")
    mlflow.log_param("forecast_validation", "full_horizon_series_only")

    mlflow.log_metric("raw_train_rows", len(train))
    mlflow.log_metric("raw_test_rows", len(test))

    # last-39 split metrics
    mlflow.log_metric("last39_train_rows", len(train_raw_part))
    mlflow.log_metric("last39_valid_rows", len(valid_raw_part))
    mlflow.log_metric("last39_train_panel_rows", len(last39_train_panel))
    mlflow.log_metric("last39_valid_panel_rows", len(last39_valid_panel))
    mlflow.log_metric("last39_full_horizon_valid_series", len(tft_last39_data["valid_dataset"]))
    mlflow.log_metric("last39_train_windows", len(tft_last39_data["train_dataset"]))

    # calendar split metrics
    mlflow.log_metric("calendar_train_rows", len(calendar_train_raw_part))
    mlflow.log_metric("calendar_valid_rows", len(calendar_valid_raw_part))
    mlflow.log_metric("calendar_train_panel_rows", len(calendar_train_panel))
    mlflow.log_metric("calendar_valid_panel_rows", len(calendar_valid_panel))
    mlflow.log_metric("calendar_full_horizon_valid_series", len(tft_calendar_data["valid_dataset"]))
    mlflow.log_metric("calendar_train_windows", len(tft_calendar_data["train_dataset"]))

    # feature dimensions
    mlflow.log_metric("num_static_cat_cols", len(tft_calendar_cols["static_cat_cols"]))
    mlflow.log_metric("num_static_real_cols", len(tft_calendar_cols["static_real_cols"]))
    mlflow.log_metric("num_known_future_reals", len(tft_calendar_cols["known_future_real_cols"]))

    preprocessing_config = {
        "model_family": "TFT",
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "primary_validation_strategy": "calendar_aligned_39_weeks",
        "secondary_validation_strategy": "last_39_weeks",
        "feature_set": "stable_no_markdowns",
        "tft_calendar_cols": tft_calendar_cols,
        "tft_last39_cols": tft_last39_cols,
        "required_base_cols": required_base_cols,
        "calendar_static_cat_cardinalities": tft_calendar_data["static_cat_cardinalities"],
        "last39_static_cat_cardinalities": tft_last39_data["static_cat_cardinalities"],
    }

    config_path = repo_root / "tft_preprocessing_config.json"

    with open(config_path, "w") as f:
        json.dump(preprocessing_config, f, indent=2)

    mlflow.log_artifact(str(config_path), artifact_path="config")
    config_path.unlink()

    tft_preprocessing_run_id = run.info.run_id

print("Logged TFT preprocessing run:", tft_preprocessing_run_id)

🏃 View run TFT_Preprocessing at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/6/runs/750382216dc54716843b78923b37770d
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/6
Logged TFT preprocessing run: 750382216dc54716843b78923b37770d


## Evaluation utilities

In [24]:
def weighted_mae_np(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    is_holiday = np.asarray(is_holiday).reshape(-1).astype(bool)

    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def inverse_scale_torch(y_scaled, target_mean, target_std):
    """
    y_scaled: [B, H]
    target_mean: [B]
    target_std: [B]
    """
    return y_scaled * target_std.unsqueeze(1) + target_mean.unsqueeze(1)


def move_batch_to_device(batch, device):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loss_fn(loss_name: str):
    loss_name = loss_name.lower()

    if loss_name == "mse":
        return nn.MSELoss()

    if loss_name in ["mae", "l1"]:
        return nn.L1Loss()

    if loss_name == "huber":
        return nn.HuberLoss(delta=1.0)

    raise ValueError(f"Unknown loss_name: {loss_name}")

## TFT Model Definitions

In [25]:
class GatedResidualBlock(nn.Module):
    """
    Lightweight gated residual block.

    This is used after attention to let the model decide how much of the
    transformed representation should be kept vs skipped.
    """
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()

        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        self.activation = nn.ELU()

    def forward(self, x):
        residual = x

        z = self.fc1(x)
        z = self.activation(z)
        z = self.dropout(z)
        z = self.fc2(z)

        gate = torch.sigmoid(self.gate(x))
        z = gate * z

        return self.norm(residual + z)

In [26]:
class StaticContextEncoder(nn.Module):
    """
    Encodes static Store/Dept/Type embeddings and static real features
    into one context vector per series.
    """
    def __init__(
        self,
        static_cat_cardinalities: list[int],
        num_static_reals: int,
        embedding_dim: int,
        hidden_dim: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.static_cat_cardinalities = static_cat_cardinalities
        self.num_static_reals = num_static_reals

        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality in static_cat_cardinalities
        ])

        input_dim = len(static_cat_cardinalities) * embedding_dim + num_static_reals

        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, static_categoricals, static_reals):
        embedded_parts = []

        for i, embedding in enumerate(self.embeddings):
            cat_values = static_categoricals[:, i].long()
            cat_values = cat_values.clamp(0, embedding.num_embeddings - 1)
            embedded_parts.append(embedding(cat_values))

        if embedded_parts:
            cat_context = torch.cat(embedded_parts, dim=1)
        else:
            cat_context = static_reals.new_zeros(static_reals.shape[0], 0)

        if self.num_static_reals > 0:
            static_input = torch.cat([cat_context, static_reals], dim=1)
        else:
            static_input = cat_context

        return self.projection(static_input)

In [27]:
class VariableSelectionNetwork(nn.Module):
    """
    Simplified TFT-style variable selection.

    Each scalar input variable is projected into hidden_dim.
    A softmax network learns feature weights at every time step.
    The output is a weighted sum of variable embeddings.
    """
    def __init__(
        self,
        num_variables: int,
        hidden_dim: int,
        context_dim: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.num_variables = num_variables
        self.hidden_dim = hidden_dim

        self.variable_projections = nn.ModuleList([
            nn.Linear(1, hidden_dim)
            for _ in range(num_variables)
        ])

        self.weight_network = nn.Sequential(
            nn.Linear(num_variables + context_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_variables),
        )

    def forward(self, x, static_context):
        """
        x: [B, T, num_variables]
        static_context: [B, context_dim]
        """
        batch_size, time_steps, num_variables = x.shape

        assert num_variables == self.num_variables

        # projected representations of each variable
        variable_embeddings = []

        for i, projection in enumerate(self.variable_projections):
            variable_i = x[:, :, i:i + 1]
            variable_embeddings.append(projection(variable_i))

        # [B, T, num_variables, hidden_dim]
        variable_embeddings = torch.stack(variable_embeddings, dim=2)

        # repeat static context across time
        static_context_time = static_context.unsqueeze(1).expand(
            -1,
            time_steps,
            -1,
        )

        weight_input = torch.cat([x, static_context_time], dim=-1)
        variable_weights = torch.softmax(
            self.weight_network(weight_input),
            dim=-1,
        )

        selected = (
            variable_weights.unsqueeze(-1)
            * variable_embeddings
        ).sum(dim=2)

        return selected

In [28]:
class TFTPointForecaster(nn.Module):
    """
    Simplified Temporal Fusion Transformer for point forecasting.

    Inputs:
        past_target:          [B, context_length]
        past_known_reals:     [B, context_length, F]
        future_known_reals:   [B, prediction_length, F]
        static_categoricals:  [B, C]
        static_reals:         [B, S]

    Output:
        forecast:             [B, prediction_length]
    """
    def __init__(
        self,
        context_length: int,
        prediction_length: int,
        static_cat_cardinalities: list[int],
        num_known_reals: int,
        num_static_reals: int,
        embedding_dim: int = 8,
        hidden_dim: int = 64,
        lstm_layers: int = 1,
        attention_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()

        if hidden_dim % attention_heads != 0:
            raise ValueError(
                f"hidden_dim={hidden_dim} must be divisible by "
                f"attention_heads={attention_heads}"
            )

        self.context_length = context_length
        self.prediction_length = prediction_length
        self.num_known_reals = num_known_reals
        self.hidden_dim = hidden_dim

        self.static_encoder = StaticContextEncoder(
            static_cat_cardinalities=static_cat_cardinalities,
            num_static_reals=num_static_reals,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        # past variables = past target + past known real covariates
        self.past_variable_selection = VariableSelectionNetwork(
            num_variables=1 + num_known_reals,
            hidden_dim=hidden_dim,
            context_dim=hidden_dim,
            dropout=dropout,
        )

        # future variables = future known real covariates
        self.future_variable_selection = VariableSelectionNetwork(
            num_variables=num_known_reals,
            hidden_dim=hidden_dim,
            context_dim=hidden_dim,
            dropout=dropout,
        )

        lstm_dropout = dropout if lstm_layers > 1 else 0.0

        self.encoder_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )

        self.decoder_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=attention_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.attention_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.static_enrichment = nn.Linear(hidden_dim, hidden_dim)
        self.static_norm = nn.LayerNorm(hidden_dim)

        self.gated_residual = GatedResidualBlock(
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        past_target,
        past_known_reals,
        future_known_reals,
        static_categoricals,
        static_reals,
    ):
        # static context: [B, hidden_dim]
        static_context = self.static_encoder(
            static_categoricals=static_categoricals,
            static_reals=static_reals,
        )

        # past input: [B, context_length, 1 + F]
        past_inputs = torch.cat(
            [
                past_target.unsqueeze(-1),
                past_known_reals,
            ],
            dim=-1,
        )

        # select/encode variables
        past_selected = self.past_variable_selection(
            past_inputs,
            static_context,
        )

        future_selected = self.future_variable_selection(
            future_known_reals,
            static_context,
        )

        # encoder over past sequence
        encoder_outputs, encoder_state = self.encoder_lstm(past_selected)

        # decoder over future known covariates, initialized from encoder state
        decoder_outputs, _ = self.decoder_lstm(
            future_selected,
            encoder_state,
        )

        # attention: future decoder steps attend over both past and future states
        temporal_memory = torch.cat(
            [encoder_outputs, decoder_outputs],
            dim=1,
        )

        attention_outputs, _ = self.attention(
            query=decoder_outputs,
            key=temporal_memory,
            value=temporal_memory,
            need_weights=False,
        )

        x = self.attention_norm(
            decoder_outputs + self.dropout(attention_outputs)
        )

        # static enrichment
        static_context_time = self.static_enrichment(static_context).unsqueeze(1)
        x = self.static_norm(x + static_context_time)

        # gated residual processing
        x = self.gated_residual(x)

        # one point forecast per future step
        forecast = self.output_head(x).squeeze(-1)

        return forecast

In [29]:
test_batch = next(iter(tft_last39_data["train_loader"]))
test_batch = move_batch_to_device(test_batch, DEVICE)

tft_test_model = TFTPointForecaster(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    static_cat_cardinalities=tft_last39_data["static_cat_cardinalities"],
    num_known_reals=tft_last39_data["num_known_reals"],
    num_static_reals=tft_last39_data["num_static_reals"],
    embedding_dim=8,
    hidden_dim=64,
    lstm_layers=1,
    attention_heads=4,
    dropout=0.1,
).to(DEVICE)

with torch.no_grad():
    test_output = tft_test_model(
        past_target=test_batch["past_target"],
        past_known_reals=test_batch["past_known_reals"],
        future_known_reals=test_batch["future_known_reals"],
        static_categoricals=test_batch["static_categoricals"],
        static_reals=test_batch["static_reals"],
    )

print("TFT output shape:", tuple(test_output.shape))
print("Expected shape:", tuple(test_batch["future_target"].shape))

assert test_output.shape == test_batch["future_target"].shape

print("Trainable parameters:", count_parameters(tft_test_model))

del tft_test_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("TFT shape test passed.")

TFT output shape: (256, 39)
Expected shape: (256, 39)
Trainable parameters: 125560
TFT shape test passed.


## Training utilities

In [30]:
def forward_model(model, batch):
    return model(
        past_target=batch["past_target"],
        past_known_reals=batch["past_known_reals"],
        future_known_reals=batch["future_known_reals"],
        static_categoricals=batch["static_categoricals"],
        static_reals=batch["static_reals"],
    )


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    total_items = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        try:
            loss = loss_fn(preds_scaled, target_scaled, batch)
        except TypeError:
            loss = loss_fn(preds_scaled, target_scaled)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

    return total_loss / total_items

In [31]:
@torch.no_grad()
def evaluate_model(model, loader, loss_fn, device, holiday_feature_idx: int):
    model.eval()

    total_loss = 0.0
    total_items = 0

    all_true = []
    all_pred = []
    all_holiday = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        try:
            loss = loss_fn(preds_scaled, target_scaled, batch)
        except TypeError:
            loss = loss_fn(preds_scaled, target_scaled)

        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]

        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())

    y_pred = np.concatenate(all_pred, axis=0)
    y_true = np.concatenate(all_true, axis=0)
    is_holiday = np.concatenate(all_holiday, axis=0)

    valid_loss = total_loss / total_items
    valid_wmae = weighted_mae_np(y_true, y_pred, is_holiday)
    valid_mae = np.mean(np.abs(y_true.reshape(-1) - y_pred.reshape(-1)))

    return {
        "valid_loss": valid_loss,
        "valid_wmae": valid_wmae,
        "valid_mae": valid_mae,
    }

In [32]:
class WeightedHuberLoss(nn.Module):
    def __init__(
        self,
        holiday_feature_idx: int,
        holiday_weight: float = 5.0,
        delta: float = 1.0,
    ):
        super().__init__()
        self.holiday_feature_idx = holiday_feature_idx
        self.holiday_weight = holiday_weight
        self.delta = delta

    def forward(self, preds_scaled, target_scaled, batch):
        element_loss = F.huber_loss(
            preds_scaled,
            target_scaled,
            reduction="none",
            delta=self.delta,
        )

        is_holiday = batch["future_known_reals"][:, :, self.holiday_feature_idx].bool()

        weights = torch.where(
            is_holiday,
            torch.full_like(element_loss, self.holiday_weight),
            torch.ones_like(element_loss),
        )

        return (weights * element_loss).sum() / weights.sum()

In [33]:
def fit_model(
    model,
    train_loader,
    valid_loader,
    optimizer,
    loss_fn,
    device,
    holiday_feature_idx: int,
    epochs: int,
    metric_prefix: str = "",
):
    best_state_dict = None
    best_valid_wmae = float("inf")
    best_epoch = None

    history = []

    prefix = f"{metric_prefix}_" if metric_prefix else ""

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
        )

        valid_metrics = evaluate_model(
            model=model,
            loader=valid_loader,
            loss_fn=loss_fn,
            device=device,
            holiday_feature_idx=holiday_feature_idx,
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **valid_metrics,
        }
        history.append(row)

        if mlflow.active_run() is not None:
            mlflow.log_metric(f"{prefix}train_loss", train_loss, step=epoch)
            mlflow.log_metric(f"{prefix}valid_loss", valid_metrics["valid_loss"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_wmae", valid_metrics["valid_wmae"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_mae", valid_metrics["valid_mae"], step=epoch)

        if valid_metrics["valid_wmae"] < best_valid_wmae:
            best_valid_wmae = valid_metrics["valid_wmae"]
            best_epoch = epoch
            best_state_dict = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.5f} | "
            f"valid_loss={valid_metrics['valid_loss']:.5f} | "
            f"valid_wmae={valid_metrics['valid_wmae']:.2f} | "
            f"valid_mae={valid_metrics['valid_mae']:.2f}"
        )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history_df = pd.DataFrame(history)

    return {
        "model": model,
        "history": history_df,
        "best_valid_wmae": best_valid_wmae,
        "best_epoch": best_epoch,
        "best_state_dict": best_state_dict,
    }

In [34]:
@torch.no_grad()
def collect_validation_predictions(
    model,
    loader,
    dataset,
    device,
    holiday_feature_idx: int,
):
    model.eval()

    all_true = []
    all_pred = []
    all_holiday = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]

        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())

    y_pred = np.concatenate(all_pred, axis=0).reshape(-1)
    y_true = np.concatenate(all_true, axis=0).reshape(-1)
    is_holiday = np.concatenate(all_holiday, axis=0).reshape(-1).astype(bool)

    index_df = dataset.get_future_index().reset_index(drop=True).copy()

    assert len(index_df) == len(y_pred)

    index_df["y_true"] = y_true
    index_df["y_pred"] = y_pred
    index_df["abs_error"] = np.abs(index_df["y_true"] - index_df["y_pred"])
    index_df["IsHoliday_eval"] = is_holiday

    return index_df

In [35]:
def plot_loss_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["valid_loss"], label="valid_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Scaled loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_wmae_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["valid_wmae"], label="valid_wmae")
    plt.xlabel("Epoch")
    plt.ylabel("Validation WMAE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

In [36]:
ARTIFACT_DIR = repo_root / "artifacts" / "tft"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

experiment_results = []

print("Artifact dir:", ARTIFACT_DIR)

Artifact dir: /kaggle/working/Walmart/artifacts/tft


## TFT Experiments: Last-39 Validation

In [ ]:
print("TFT last-39 train windows:", len(tft_last39_data["train_dataset"]))
print("TFT last-39 valid series:", len(tft_last39_data["valid_dataset"]))
print("TFT last-39 train batches:", len(tft_last39_data["train_loader"]))
print("TFT last-39 valid batches:", len(tft_last39_data["valid_loader"]))

print("\nTFT calendar train windows:", len(tft_calendar_data["train_dataset"]))
print("TFT calendar valid series:", len(tft_calendar_data["valid_dataset"]))
print("TFT calendar train batches:", len(tft_calendar_data["train_loader"]))
print("TFT calendar valid batches:", len(tft_calendar_data["valid_loader"]))

TFT last-39 train windows: 38464
TFT last-39 valid series: 2762
TFT last-39 train batches: 151
TFT last-39 valid batches: 11

TFT calendar train windows: 2683
TFT calendar valid series: 2762
TFT calendar train batches: 11
TFT calendar valid batches: 11


In [ ]:
TFT_BASELINE_CONFIG = {
    "model": "TFTPointForecaster",
    "model_variant": "tft_point",
    "feature_set": "stable_no_markdowns",
    "validation_strategy": "last_39_weeks",
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "embedding_dim": 8,
    "hidden_dim": 64,
    "lstm_layers": 1,
    "attention_heads": 4,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "batch_size": BATCH_SIZE,
    "epochs": 10,
    "optimizer": "AdamW",
}

In [ ]:
set_seed(SEED)

config = TFT_BASELINE_CONFIG.copy()

run_name = (
    f"TFT_Last39_Baseline_"
    f"h{config['hidden_dim']}_"
    f"emb{config['embedding_dim']}_"
    f"heads{config['attention_heads']}_"
    f"drop{config['dropout']}_"
    f"lr{config['lr']}_"
    f"wd{config['weight_decay']}_"
    f"{config['loss_name']}"
)

print(run_name)
print(config)

model = TFTPointForecaster(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    static_cat_cardinalities=tft_last39_data["static_cat_cardinalities"],
    num_known_reals=tft_last39_data["num_known_reals"],
    num_static_reals=tft_last39_data["num_static_reals"],
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    lstm_layers=config["lstm_layers"],
    attention_heads=config["attention_heads"],
    dropout=config["dropout"],
).to(DEVICE)

print(model)
print("Trainable parameters:", count_parameters(model))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"],
)

loss_fn = make_loss_fn(config["loss_name"])

with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id

    mlflow.log_params({
        **config,
        "train_loader_type": "FastTensorDataLoader",
        "train_windows": len(tft_last39_data["train_dataset"]),
        "valid_series": len(tft_last39_data["valid_dataset"]),
        "num_known_reals": tft_last39_data["num_known_reals"],
        "num_static_reals": tft_last39_data["num_static_reals"],
        "static_cat_cardinalities": str(tft_last39_data["static_cat_cardinalities"]),
        "trainable_parameters": count_parameters(model),
    })

    result = fit_model(
        model=model,
        train_loader=tft_last39_data["train_loader"],
        valid_loader=tft_last39_data["valid_loader"],
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        holiday_feature_idx=tft_last39_data["holiday_feature_idx"],
        epochs=config["epochs"],
        metric_prefix="last39",
    )

    mlflow.log_metric("last39_best_valid_wmae", result["best_valid_wmae"])
    mlflow.log_metric("last39_best_epoch", result["best_epoch"])

print("Run ID:", run_id)
print("Best last-39 WMAE:", result["best_valid_wmae"])
print("Best epoch:", result["best_epoch"])

tft_baseline_result = {
    "run_name": run_name,
    "run_id": run_id,
    **config,
    "best_valid_wmae": result["best_valid_wmae"],
    "best_epoch": result["best_epoch"],
}

display(result["history"])

TFT_Last39_Baseline_h64_emb8_heads4_drop0.1_lr0.001_wd0.0001_huber
{'model': 'TFTPointForecaster', 'model_variant': 'tft_point', 'feature_set': 'stable_no_markdowns', 'validation_strategy': 'last_39_weeks', 'context_length': 52, 'prediction_length': 39, 'embedding_dim': 8, 'hidden_dim': 64, 'lstm_layers': 1, 'attention_heads': 4, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber', 'batch_size': 256, 'epochs': 10, 'optimizer': 'AdamW'}
TFTPointForecaster(
  (static_encoder): StaticContextEncoder(
    (embeddings): ModuleList(
      (0): Embedding(46, 8)
      (1): Embedding(82, 8)
      (2): Embedding(4, 8)
    )
    (projection): Sequential(
      (0): Linear(in_features=25, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (past_variable_selection): VariableSelectionNetwork(
    (variable_projections): ModuleList(
      (0-11): 12 x Linear(in_features=1

,epoch,train_loss,valid_loss,valid_wmae,valid_mae
0,1,0.275557,0.447509,2662.166752,2599.398193
1,2,0.214731,0.428306,2547.024190,2491.294434
2,3,0.192998,0.409223,2439.410944,2395.804199
3,4,0.179812,0.406909,2427.514821,2387.561768
4,5,0.170503,0.407423,2407.186227,2369.737061
5,6,0.164244,0.416390,2441.871263,2410.738037
6,7,0.159059,0.404689,2376.386930,2334.919434
7,8,0.155391,0.403962,2370.955287,2327.432129
8,9,0.152480,0.409542,2398.136043,2358.740234
9,10,0.148995,0.402356,2324.882614,2310.903564


In [ ]:
TFT_LAST39_GRID = [
    # baseline family
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # lower learning rate family
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly faster learning
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 2e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # larger hidden dimension
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # larger embeddings
    {
        "embedding_dim": 16,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 16,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 16,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # 8 attention heads for hidden_dim 128
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 8,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 16,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 8,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # stronger weight decay
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },
]

EPOCHS_LAST39_GRID = 25

print("Number of TFT last-39 grid configs:", len(TFT_LAST39_GRID))
print("Epochs per config:", EPOCHS_LAST39_GRID)
print("Approx total epochs:", len(TFT_LAST39_GRID) * EPOCHS_LAST39_GRID)

Number of TFT last-39 grid configs: 16
Epochs per config: 25
Approx total epochs: 400


In [ ]:
set_seed(SEED)

tft_last39_results = []

START_FROM_CONFIG = 1

for i, config in enumerate(TFT_LAST39_GRID, start=1):
    if i < START_FROM_CONFIG:
        continue
    run_name = (
        f"TFT_Last39_"
        f"h{config['hidden_dim']}_"
        f"emb{config['embedding_dim']}_"
        f"heads{config['attention_heads']}_"
        f"layers{config['lstm_layers']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Run {i}/{len(TFT_LAST39_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + i)

    model = TFTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=tft_last39_data["static_cat_cardinalities"],
        num_known_reals=tft_last39_data["num_known_reals"],
        num_static_reals=tft_last39_data["num_static_reals"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        lstm_layers=config["lstm_layers"],
        attention_heads=config["attention_heads"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)
    print("Trainable parameters:", n_params)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "TFTPointForecaster",
            "model_variant": "tft_point",
            "feature_set": "stable_no_markdowns",
            "validation_strategy": "last_39_weeks",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "lstm_layers": config["lstm_layers"],
            "attention_heads": config["attention_heads"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_LAST39_GRID,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(tft_last39_data["train_dataset"]),
            "valid_series": len(tft_last39_data["valid_dataset"]),
            "num_known_reals": tft_last39_data["num_known_reals"],
            "num_static_reals": tft_last39_data["num_static_reals"],
            "static_cat_cardinalities": str(tft_last39_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
        })

        result = fit_model(
            model=model,
            train_loader=tft_last39_data["train_loader"],
            valid_loader=tft_last39_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=tft_last39_data["holiday_feature_idx"],
            epochs=EPOCHS_LAST39_GRID,
            metric_prefix="last39",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("last39_best_valid_wmae", best_wmae)
        mlflow.log_metric("last39_best_epoch", best_epoch)

    tft_last39_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "trainable_parameters": n_params,
        "last39_best_valid_wmae": best_wmae,
        "last39_best_epoch": best_epoch,
    })

    print(f"Best last-39 WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

tft_last39_results_df = (
    pd.DataFrame(tft_last39_results)
    .sort_values("last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_last39_results_df)

last39_results_path = ARTIFACT_DIR / "tft_last39_grid_results.csv"
tft_last39_results_df.to_csv(last39_results_path, index=False)

print("Saved:", last39_results_path)

Run 1/16: TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0.001_wd0.0001_huber
{'embedding_dim': 8, 'hidden_dim': 64, 'lstm_layers': 1, 'attention_heads': 4, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Trainable parameters: 125560
Epoch 001 | train_loss=0.27335 | valid_loss=0.43548 | valid_wmae=2693.53 | valid_mae=2614.42
Epoch 001 | train_loss=0.27335 | valid_loss=0.43548 | valid_wmae=2693.53 | valid_mae=2614.42
Epoch 002 | train_loss=0.21364 | valid_loss=0.41485 | valid_wmae=2500.41 | valid_mae=2426.53
Epoch 002 | train_loss=0.21364 | valid_loss=0.41485 | valid_wmae=2500.41 | valid_mae=2426.53
Epoch 003 | train_loss=0.19016 | valid_loss=0.40301 | valid_wmae=2432.54 | valid_mae=2363.41
Epoch 003 | train_loss=0.19016 | valid_loss=0.40301 | valid_wmae=2432.54 | valid_mae=2363.41
Epoch 004 | train_loss=0.17637 | valid_loss=0.41152 | valid_wmae=2429.87 | valid_mae=2387.34
Epoch 004 | train_loss=0.17637 | valid_loss=0.41152 | valid_wmae=2429.87 | valid_mae=2387.

,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,trainable_parameters,last39_best_valid_wmae,last39_best_epoch
0,TFT_Last39_h64_emb8_heads4_layers1_drop0.3_lr0...,ce7943e5789c4d0197f9da542266e435,8,64,1,4,0.3,0.0010,0.0001,huber,125560,2223.957789,13
1,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,38b0bc3c414c4fd2911f818f8af4d725,8,64,1,4,0.2,0.0010,0.0001,huber,125560,2248.005916,23
2,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,d8efc874e5f441e7983f50c7f2ca0735,8,64,1,4,0.1,0.0010,0.0001,huber,125560,2248.751290,15
3,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,2bb0b16b545649e8917ef17b38cfdcd4,8,64,1,4,0.2,0.0010,0.0005,huber,125560,2253.931945,17
4,TFT_Last39_h64_emb16_heads4_layers1_drop0.2_lr...,4a018b71cf0c413cbd22c8267cb819e6,16,64,1,4,0.2,0.0010,0.0001,huber,128152,2259.420502,17
5,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,11cadfcef12949958d419608f8ef56a2,8,128,1,4,0.2,0.0010,0.0001,huber,479416,2263.946137,8
6,TFT_Last39_h128_emb8_heads4_layers1_drop0.3_lr...,e08cf77ed7a24dc397b7d3d3dee3c0e2,8,128,1,4,0.3,0.0005,0.0001,huber,479416,2266.685444,12
7,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,6589796b0f684e6ca269a7522a2b8eca,8,128,1,4,0.2,0.0005,0.0005,huber,479416,2275.846929,25
8,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,1a29140ba3db4fb68983990cb6641e73,8,64,1,4,0.1,0.0005,0.0001,huber,125560,2281.499444,21
9,TFT_Last39_h64_emb16_heads4_layers1_drop0.1_lr...,267777d4121747f29f72140dbd2d82ff,16,64,1,4,0.1,0.0010,0.0001,huber,128152,2285.178970,25


Saved: /content/drive/MyDrive/Machine Learning/Walmart/artifacts/tft/tft_last39_grid_results.csv


,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,trainable_parameters,last39_best_valid_wmae,last39_best_epoch
0,TFT_Last39_h64_emb8_heads4_layers1_drop0.3_lr0...,ce7943e5789c4d0197f9da542266e435,8,64,1,4,0.3,0.0010,0.0001,huber,125560,2223.957789,13
1,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,38b0bc3c414c4fd2911f818f8af4d725,8,64,1,4,0.2,0.0010,0.0001,huber,125560,2248.005916,23
2,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,d8efc874e5f441e7983f50c7f2ca0735,8,64,1,4,0.1,0.0010,0.0001,huber,125560,2248.751290,15
3,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,2bb0b16b545649e8917ef17b38cfdcd4,8,64,1,4,0.2,0.0010,0.0005,huber,125560,2253.931945,17
4,TFT_Last39_h64_emb16_heads4_layers1_drop0.2_lr...,4a018b71cf0c413cbd22c8267cb819e6,16,64,1,4,0.2,0.0010,0.0001,huber,128152,2259.420502,17
5,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,11cadfcef12949958d419608f8ef56a2,8,128,1,4,0.2,0.0010,0.0001,huber,479416,2263.946137,8
6,TFT_Last39_h128_emb8_heads4_layers1_drop0.3_lr...,e08cf77ed7a24dc397b7d3d3dee3c0e2,8,128,1,4,0.3,0.0005,0.0001,huber,479416,2266.685444,12
7,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,6589796b0f684e6ca269a7522a2b8eca,8,128,1,4,0.2,0.0005,0.0005,huber,479416,2275.846929,25
8,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,1a29140ba3db4fb68983990cb6641e73,8,64,1,4,0.1,0.0005,0.0001,huber,125560,2281.499444,21
9,TFT_Last39_h64_emb16_heads4_layers1_drop0.1_lr...,267777d4121747f29f72140dbd2d82ff,16,64,1,4,0.1,0.0010,0.0001,huber,128152,2285.178970,25


Saved: /content/drive/MyDrive/Machine Learning/Walmart/artifacts/tft/tft_last39_grid_results.csv


In [ ]:
top_configs = (
    tft_last39_results_df
    .sort_values("last39_best_valid_wmae")
    .head(8)
    .copy()
)

diverse_configs = []

# best hidden_dim 64
diverse_configs.append(
    tft_last39_results_df[
        tft_last39_results_df["hidden_dim"] == 64
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best hidden_dim 128
diverse_configs.append(
    tft_last39_results_df[
        tft_last39_results_df["hidden_dim"] == 128
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best higher dropout
diverse_configs.append(
    tft_last39_results_df[
        tft_last39_results_df["dropout"] >= 0.2
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best low learning rate
diverse_configs.append(
    tft_last39_results_df[
        tft_last39_results_df["lr"] <= 5e-4
    ].sort_values("last39_best_valid_wmae").head(1)
)

tft_calendar_candidate_configs = pd.concat(
    [top_configs] + diverse_configs,
    axis=0,
).drop_duplicates(
    subset=[
        "embedding_dim",
        "hidden_dim",
        "lstm_layers",
        "attention_heads",
        "dropout",
        "lr",
        "weight_decay",
        "loss_name",
    ]
).reset_index(drop=True)

display(tft_calendar_candidate_configs)
print("Calendar candidate configs:", len(tft_calendar_candidate_configs))

,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,trainable_parameters,last39_best_valid_wmae,last39_best_epoch
0,TFT_Last39_h64_emb8_heads4_layers1_drop0.3_lr0...,ce7943e5789c4d0197f9da542266e435,8,64,1,4,0.3,0.0010,0.0001,huber,125560,2223.957789,13
1,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,38b0bc3c414c4fd2911f818f8af4d725,8,64,1,4,0.2,0.0010,0.0001,huber,125560,2248.005916,23
2,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,d8efc874e5f441e7983f50c7f2ca0735,8,64,1,4,0.1,0.0010,0.0001,huber,125560,2248.751290,15
3,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,2bb0b16b545649e8917ef17b38cfdcd4,8,64,1,4,0.2,0.0010,0.0005,huber,125560,2253.931945,17
4,TFT_Last39_h64_emb16_heads4_layers1_drop0.2_lr...,4a018b71cf0c413cbd22c8267cb819e6,16,64,1,4,0.2,0.0010,0.0001,huber,128152,2259.420502,17
5,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,11cadfcef12949958d419608f8ef56a2,8,128,1,4,0.2,0.0010,0.0001,huber,479416,2263.946137,8
6,TFT_Last39_h128_emb8_heads4_layers1_drop0.3_lr...,e08cf77ed7a24dc397b7d3d3dee3c0e2,8,128,1,4,0.3,0.0005,0.0001,huber,479416,2266.685444,12
7,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,6589796b0f684e6ca269a7522a2b8eca,8,128,1,4,0.2,0.0005,0.0005,huber,479416,2275.846929,25


Calendar candidate configs: 8


## TFT Experiments: Calendar-aligned

In [ ]:
EPOCHS_CALENDAR_CHECK = 50

tft_calendar_results = []

TFT_CALENDAR_EXTRA_GRID = [
    # current best, trained longer
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly lower LR for smoother late training
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly higher LR, still conservative
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 7e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly less dropout
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.25,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly more dropout
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.35,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # stronger weight decay with best family
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },

    # more attention heads for h128
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 8,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly larger embedding, same best family
    {
        "embedding_dim": 16,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # recheck strong h64 family longer
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
]

EPOCHS_CALENDAR_EXTRA = 100

print("Extra calendar configs:", len(TFT_CALENDAR_EXTRA_GRID))
print("Epochs per config:", EPOCHS_CALENDAR_EXTRA)

START_FROM_CONFIG = 2

# for i, row in tft_calendar_candidate_configs.iterrows():
for i, row in enumerate(TFT_CALENDAR_EXTRA_GRID, start=1):
    if i < START_FROM_CONFIG:
        continue
    config = {
        "embedding_dim": int(row["embedding_dim"]),
        "hidden_dim": int(row["hidden_dim"]),
        "lstm_layers": int(row["lstm_layers"]),
        "attention_heads": int(row["attention_heads"]),
        "dropout": float(row["dropout"]),
        "lr": float(row["lr"]),
        "weight_decay": float(row["weight_decay"]),
        "loss_name": row["loss_name"],
    }

    run_name = (
        f"TFT_CalendarAligned_"
        f"h{config['hidden_dim']}_"
        f"emb{config['embedding_dim']}_"
        f"heads{config['attention_heads']}_"
        f"layers{config['lstm_layers']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Calendar run {i}/{len(tft_calendar_candidate_configs)}: {run_name}")
    print(config)

    set_seed(SEED + 100 + i)

    model = TFTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=tft_calendar_data["static_cat_cardinalities"],
        num_known_reals=tft_calendar_data["num_known_reals"],
        num_static_reals=tft_calendar_data["num_static_reals"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        lstm_layers=config["lstm_layers"],
        attention_heads=config["attention_heads"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)
    print("Trainable parameters:", n_params)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "TFTPointForecaster",
            "model_variant": "tft_point",
            "feature_set": "stable_no_markdowns",
            "validation_strategy": "calendar_aligned_39_weeks",
            "calendar_valid_start": "2011-11-04",
            "calendar_valid_end": "2012-07-27",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "lstm_layers": config["lstm_layers"],
            "attention_heads": config["attention_heads"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            # "epochs": EPOCHS_CALENDAR_CHECK,
            "epochs": EPOCHS_CALENDAR_EXTRA,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(tft_calendar_data["train_dataset"]),
            "valid_series": len(tft_calendar_data["valid_dataset"]),
            "num_known_reals": tft_calendar_data["num_known_reals"],
            "num_static_reals": tft_calendar_data["num_static_reals"],
            "static_cat_cardinalities": str(tft_calendar_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
            # "source_last39_run_id": row["run_id"],
            # "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
        })

        result = fit_model(
            model=model,
            train_loader=tft_calendar_data["train_loader"],
            valid_loader=tft_calendar_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=tft_calendar_data["holiday_feature_idx"],
            epochs=EPOCHS_CALENDAR_CHECK,
            metric_prefix="calendar",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("calendar_best_valid_wmae", best_wmae)
        mlflow.log_metric("calendar_best_epoch", best_epoch)

    tft_calendar_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "trainable_parameters": n_params,
        # "source_last39_run_id": row["run_id"],
        # "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
        "calendar_best_valid_wmae": best_wmae,
        "calendar_best_epoch": best_epoch,
    })

    print(f"Best calendar WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

tft_calendar_results_df = (
    pd.DataFrame(tft_calendar_results)
    .sort_values("calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_calendar_results_df)

calendar_results_path = ARTIFACT_DIR / "tft_calendar_check_results.csv"
tft_calendar_results_df.to_csv(calendar_results_path, index=False)

print("Saved:", calendar_results_path)

Extra calendar configs: 9
Epochs per config: 100
Calendar run 2/8: TFT_CalendarAligned_h128_emb8_heads4_layers1_drop0.3_lr0.0003_wd0.0001_huber
{'embedding_dim': 8, 'hidden_dim': 128, 'lstm_layers': 1, 'attention_heads': 4, 'dropout': 0.3, 'lr': 0.0003, 'weight_decay': 0.0001, 'loss_name': 'huber'}
Trainable parameters: 479416
Epoch 001 | train_loss=0.30484 | valid_loss=0.55007 | valid_wmae=3680.03 | valid_mae=3180.74
Epoch 002 | train_loss=0.29514 | valid_loss=0.54665 | valid_wmae=3666.30 | valid_mae=3157.23
Epoch 003 | train_loss=0.28787 | valid_loss=0.55763 | valid_wmae=3703.87 | valid_mae=3171.01
Epoch 004 | train_loss=0.27202 | valid_loss=0.59239 | valid_wmae=3861.88 | valid_mae=3320.83
Epoch 005 | train_loss=0.25653 | valid_loss=0.57591 | valid_wmae=3744.82 | valid_mae=3238.71
Epoch 006 | train_loss=0.24992 | valid_loss=0.57916 | valid_wmae=3725.36 | valid_mae=3231.77
Epoch 007 | train_loss=0.24658 | valid_loss=0.57874 | valid_wmae=3709.21 | valid_mae=3225.14
Epoch 008 | train_lo

,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,trainable_parameters,calendar_best_valid_wmae,calendar_best_epoch
0,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,7b682477344b4601894c303c427352a0,8,128,1,4,0.30,0.0003,0.0001,huber,479416,3414.541303,47
1,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,df81a158719444c084b2800ec0c96f64,8,128,1,4,0.30,0.0005,0.0005,huber,479416,3418.171710,41
2,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,16b7cbfaec9f4651b3b8325b6f024ee9,8,128,1,4,0.30,0.0007,0.0001,huber,479416,3425.109727,33
3,TFT_CalendarAligned_h128_emb8_heads8_layers1_d...,bdbdc1a31fa34b4a853a940951b390bc,8,128,1,8,0.30,0.0005,0.0001,huber,479416,3445.481535,26
4,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,31cebbe95d554fa2adfff27a58acc27a,8,128,1,4,0.25,0.0005,0.0001,huber,479416,3489.906193,31
5,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,c3d3d5bb67e74b6e8c4816962a7f7b8b,8,128,1,4,0.35,0.0005,0.0001,huber,479416,3495.364657,38
6,TFT_CalendarAligned_h64_emb8_heads4_layers1_dr...,a48396e70bc44e4d9e85032240667894,8,64,1,4,0.10,0.0010,0.0001,huber,125560,3515.131542,25
7,TFT_CalendarAligned_h128_emb16_heads4_layers1_...,55844240980742c4b4311673421ff210,16,128,1,4,0.30,0.0005,0.0001,huber,483544,3542.305381,40


Saved: /content/drive/MyDrive/Machine Learning/Walmart/artifacts/tft/tft_calendar_check_results.csv


In [ ]:
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    output_format="pandas",
)

calendar_runs = runs[
    runs["tags.mlflow.runName"].astype(str).str.contains("TFT_CalendarAligned", na=False)
].copy()

cols = [
    "tags.mlflow.runName",
    "run_id",
    "params.hidden_dim",
    "params.embedding_dim",
    "params.attention_heads",
    "params.lstm_layers",
    "params.dropout",
    "params.learning_rate",
    "params.weight_decay",
    "params.loss_name",
    "params.source_last39_run_id",
    "params.source_last39_best_valid_wmae",
    "metrics.calendar_best_valid_wmae",
    "metrics.calendar_best_epoch",
]

existing_cols = [c for c in cols if c in calendar_runs.columns]

tft_calendar_results_df = (
    calendar_runs[existing_cols]
    .dropna(subset=["metrics.calendar_best_valid_wmae"])
    .sort_values("metrics.calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_calendar_results_df)

,tags.mlflow.runName,run_id,params.hidden_dim,params.embedding_dim,params.attention_heads,params.lstm_layers,params.dropout,params.learning_rate,params.weight_decay,params.loss_name,params.source_last39_run_id,params.source_last39_best_valid_wmae,metrics.calendar_best_valid_wmae,metrics.calendar_best_epoch
0,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,d513e34633194128a1f4e970ebc64f80,128,8,4,1,0.3,0.0005,0.0001,huber,e08cf77ed7a24dc397b7d3d3dee3c0e2,2266.6854437496913,3414.375399,47.0
1,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,7b682477344b4601894c303c427352a0,128,8,4,1,0.3,0.0003,0.0001,huber,None,None,3414.541303,47.0
2,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,df81a158719444c084b2800ec0c96f64,128,8,4,1,0.3,0.0005,0.0005,huber,None,None,3418.171710,41.0
3,TFT_CalendarAligned_h64_emb8_heads4_layers1_dr...,e952412ca3ee46a8979a56ded212f36d,64,8,4,1,0.1,0.001,0.0001,huber,d8efc874e5f441e7983f50c7f2ca0735,2248.751289535704,3421.152823,23.0
4,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,16b7cbfaec9f4651b3b8325b6f024ee9,128,8,4,1,0.3,0.0007,0.0001,huber,None,None,3425.109727,33.0
5,TFT_CalendarAligned_h64_emb8_heads4_layers1_dr...,41955c9d49f04b7c86e3fac624ca0ceb,64,8,4,1,0.2,0.001,0.0001,huber,38b0bc3c414c4fd2911f818f8af4d725,2248.0059164111435,3443.924470,39.0
6,TFT_CalendarAligned_h128_emb8_heads8_layers1_d...,bdbdc1a31fa34b4a853a940951b390bc,128,8,8,1,0.3,0.0005,0.0001,huber,None,None,3445.481535,26.0
7,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,33d81e374b9f49f68c8862a1fa07e84d,128,8,4,1,0.2,0.0005,0.0005,huber,6589796b0f684e6ca269a7522a2b8eca,2275.846928820417,3450.989590,23.0
8,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,7b256d886b994c0b89cd41ebe2d8a911,128,8,4,1,0.3,0.0005,0.0001,huber,None,None,3455.569266,24.0
9,TFT_CalendarAligned_h64_emb16_heads4_layers1_d...,26d645e931754c8b9c463bb4ec6e7a66,64,16,4,1,0.2,0.001,0.0001,huber,4a018b71cf0c413cbd22c8267cb819e6,2259.4205019218775,3457.110001,16.0


### Weighted loss


In [ ]:
TFT_WEIGHTED_HUBER_GRID = [
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },
]

EPOCHS_WEIGHTED_HUBER = 100

In [ ]:
set_seed(SEED)

tft_weighted_huber_results = []

for i, config in enumerate(TFT_WEIGHTED_HUBER_GRID, start=1):
    run_name = (
        f"TFT_CalendarWeightedHuber_"
        f"h{config['hidden_dim']}_"
        f"emb{config['embedding_dim']}_"
        f"heads{config['attention_heads']}_"
        f"layers{config['lstm_layers']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"hw{config['holiday_weight']}"
    )

    print("=" * 100)
    print(f"Weighted Huber run {i}/{len(TFT_WEIGHTED_HUBER_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + 300 + i)

    model = TFTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=tft_calendar_data["static_cat_cardinalities"],
        num_known_reals=tft_calendar_data["num_known_reals"],
        num_static_reals=tft_calendar_data["num_static_reals"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        lstm_layers=config["lstm_layers"],
        attention_heads=config["attention_heads"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)
    print("Trainable parameters:", n_params)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = WeightedHuberLoss(
        holiday_feature_idx=tft_calendar_data["holiday_feature_idx"],
        holiday_weight=config["holiday_weight"],
        delta=config["delta"],
    )

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "TFTPointForecaster",
            "model_variant": "tft_point",
            "feature_set": "stable_no_markdowns",
            "validation_strategy": "calendar_aligned_39_weeks",
            "calendar_valid_start": "2011-11-04",
            "calendar_valid_end": "2012-07-27",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "lstm_layers": config["lstm_layers"],
            "attention_heads": config["attention_heads"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": "weighted_huber",
            "holiday_weight": config["holiday_weight"],
            "huber_delta": config["delta"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_WEIGHTED_HUBER,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(tft_calendar_data["train_dataset"]),
            "valid_series": len(tft_calendar_data["valid_dataset"]),
            "num_known_reals": tft_calendar_data["num_known_reals"],
            "num_static_reals": tft_calendar_data["num_static_reals"],
            "static_cat_cardinalities": str(tft_calendar_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
        })

        result = fit_model(
            model=model,
            train_loader=tft_calendar_data["train_loader"],
            valid_loader=tft_calendar_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=tft_calendar_data["holiday_feature_idx"],
            epochs=EPOCHS_WEIGHTED_HUBER,
            metric_prefix="calendar_weighted",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("calendar_best_valid_wmae", best_wmae)
        mlflow.log_metric("calendar_best_epoch", best_epoch)

    tft_weighted_huber_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "trainable_parameters": n_params,
        "calendar_best_valid_wmae": best_wmae,
        "calendar_best_epoch": best_epoch,
    })

    print(f"Best weighted calendar WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

tft_weighted_huber_results_df = (
    pd.DataFrame(tft_weighted_huber_results)
    .sort_values("calendar_weighted_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_weighted_huber_results_df)

Weighted Huber run 1/3: TFT_CalendarWeightedHuber_h128_emb8_heads4_layers1_drop0.3_lr0.0005_wd0.0001_hw5.0
{'embedding_dim': 8, 'hidden_dim': 128, 'lstm_layers': 1, 'attention_heads': 4, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'loss_name': 'weighted_huber', 'holiday_weight': 5.0, 'delta': 1.0}
Trainable parameters: 479416
Epoch 001 | train_loss=0.31872 | valid_loss=0.61352 | valid_wmae=3664.80 | valid_mae=3155.80
Epoch 002 | train_loss=0.29694 | valid_loss=0.61872 | valid_wmae=3699.61 | valid_mae=3176.60
Epoch 003 | train_loss=0.27015 | valid_loss=0.66478 | valid_wmae=3885.17 | valid_mae=3360.65
Epoch 004 | train_loss=0.25912 | valid_loss=0.64639 | valid_wmae=3775.21 | valid_mae=3275.25
Epoch 005 | train_loss=0.25330 | valid_loss=0.63838 | valid_wmae=3732.83 | valid_mae=3246.87
Epoch 006 | train_loss=0.24982 | valid_loss=0.63363 | valid_wmae=3710.21 | valid_mae=3232.69
Epoch 007 | train_loss=0.24685 | valid_loss=0.62307 | valid_wmae=3663.80 | valid_mae=3188.52
Epoch 008 |

,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,holiday_weight,delta,trainable_parameters,calendar_weighted_best_valid_wmae,calendar_weighted_best_epoch
0,TFT_CalendarWeightedHuber_h128_emb8_heads4_lay...,81ab89771fb44beda527234195453197,8,128,1,4,0.3,0.0005,0.0001,weighted_huber,5.0,1.0,479416,3401.245534,40
1,TFT_CalendarWeightedHuber_h64_emb8_heads4_laye...,7dbd639efe344e8ab8f970c534899210,8,64,1,4,0.1,0.0010,0.0001,weighted_huber,5.0,1.0,125560,3436.693319,74
2,TFT_CalendarWeightedHuber_h128_emb8_heads4_lay...,cf421a2dd2dd40daa242089cacb48eb4,8,128,1,4,0.3,0.0003,0.0001,weighted_huber,5.0,1.0,479416,3467.960446,79


In [ ]:
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    output_format="pandas",
)

calendar_runs = runs[
    runs["tags.mlflow.runName"].astype(str).str.contains("TFT_Calendar", na=False)
].copy()

cols = [
    "tags.mlflow.runName",
    "run_id",
    "params.hidden_dim",
    "params.embedding_dim",
    "params.attention_heads",
    "params.lstm_layers",
    "params.dropout",
    "params.learning_rate",
    "params.weight_decay",
    "params.loss_name",
    "params.holiday_weight",
    "metrics.calendar_best_valid_wmae",
    "metrics.calendar_best_epoch",
]

existing_cols = [c for c in cols if c in calendar_runs.columns]

tft_calendar_all_results_df = (
    calendar_runs[existing_cols]
    .dropna(subset=["metrics.calendar_best_valid_wmae"])
    .sort_values("metrics.calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_calendar_all_results_df)

,tags.mlflow.runName,run_id,params.hidden_dim,params.embedding_dim,params.attention_heads,params.lstm_layers,params.dropout,params.learning_rate,params.weight_decay,params.loss_name,params.holiday_weight,metrics.calendar_best_valid_wmae,metrics.calendar_best_epoch
0,TFT_CalendarWeightedHuber_h128_emb8_heads4_lay...,81ab89771fb44beda527234195453197,128,8,4,1,0.3,0.0005,0.0001,weighted_huber,5.0,3401.245534,40.0
1,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,d513e34633194128a1f4e970ebc64f80,128,8,4,1,0.3,0.0005,0.0001,huber,None,3414.375399,47.0
2,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,7b682477344b4601894c303c427352a0,128,8,4,1,0.3,0.0003,0.0001,huber,None,3414.541303,47.0
3,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,df81a158719444c084b2800ec0c96f64,128,8,4,1,0.3,0.0005,0.0005,huber,None,3418.171710,41.0
4,TFT_CalendarAligned_h64_emb8_heads4_layers1_dr...,e952412ca3ee46a8979a56ded212f36d,64,8,4,1,0.1,0.001,0.0001,huber,None,3421.152823,23.0
5,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,16b7cbfaec9f4651b3b8325b6f024ee9,128,8,4,1,0.3,0.0007,0.0001,huber,None,3425.109727,33.0
6,TFT_CalendarWeightedHuber_h64_emb8_heads4_laye...,7dbd639efe344e8ab8f970c534899210,64,8,4,1,0.1,0.001,0.0001,weighted_huber,5.0,3436.693319,74.0
7,TFT_CalendarAligned_h64_emb8_heads4_layers1_dr...,41955c9d49f04b7c86e3fac624ca0ceb,64,8,4,1,0.2,0.001,0.0001,huber,None,3443.924470,39.0
8,TFT_CalendarAligned_h128_emb8_heads8_layers1_d...,bdbdc1a31fa34b4a853a940951b390bc,128,8,8,1,0.3,0.0005,0.0001,huber,None,3445.481535,26.0
9,TFT_CalendarAligned_h128_emb8_heads4_layers1_d...,33d81e374b9f49f68c8862a1fa07e84d,128,8,4,1,0.2,0.0005,0.0005,huber,None,3450.989590,23.0


In [ ]:
TFT_LAST39_WEIGHTED_GRID = [
    # exact strong h64 family changing loss to weighted_huber
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },

    # eame family, slightly less dropout
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },

    # same family, light dropout
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },

    # more conservative LR, same current best Kaggle architecture
    {
        "embedding_dim": 8,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },

    # slightly larger embedding, still h64
    {
        "embedding_dim": 16,
        "hidden_dim": 64,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },

    # one h128 comparison, but not the main focus
    {
        "embedding_dim": 8,
        "hidden_dim": 128,
        "lstm_layers": 1,
        "attention_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "weighted_huber",
        "holiday_weight": 5.0,
        "delta": 1.0,
    },
]

EPOCHS_LAST39_WEIGHTED = 35

print("Last-39 weighted configs:", len(TFT_LAST39_WEIGHTED_GRID))
print("Epochs per config:", EPOCHS_LAST39_WEIGHTED)

Last-39 weighted configs: 6
Epochs per config: 35


In [ ]:
set_seed(SEED)

tft_last39_weighted_results = []

START_FROM_CONFIG = 6

for i, config in enumerate(TFT_LAST39_WEIGHTED_GRID, start=1):
    if i < START_FROM_CONFIG:
        continue
    run_name = (
        f"TFT_Last39WeightedHuber_"
        f"h{config['hidden_dim']}_"
        f"emb{config['embedding_dim']}_"
        f"heads{config['attention_heads']}_"
        f"layers{config['lstm_layers']}_"
        f"drop{config['dropout']}_"
        f"lr{config['lr']}_"
        f"wd{config['weight_decay']}_"
        f"hw{config['holiday_weight']}"
    )

    print("=" * 100)
    print(f"Last-39 weighted run {i}/{len(TFT_LAST39_WEIGHTED_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + 500 + i)

    model = TFTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=tft_last39_data["static_cat_cardinalities"],
        num_known_reals=tft_last39_data["num_known_reals"],
        num_static_reals=tft_last39_data["num_static_reals"],
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        lstm_layers=config["lstm_layers"],
        attention_heads=config["attention_heads"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)
    print("Trainable parameters:", n_params)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = WeightedHuberLoss(
        holiday_feature_idx=tft_last39_data["holiday_feature_idx"],
        holiday_weight=config["holiday_weight"],
        delta=config["delta"],
    )

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "TFTPointForecaster",
            "model_variant": "tft_point",
            "feature_set": "stable_no_markdowns",
            "validation_strategy": "last_39_weeks",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "embedding_dim": config["embedding_dim"],
            "hidden_dim": config["hidden_dim"],
            "lstm_layers": config["lstm_layers"],
            "attention_heads": config["attention_heads"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": "weighted_huber",
            "holiday_weight": config["holiday_weight"],
            "huber_delta": config["delta"],
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS_LAST39_WEIGHTED,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(tft_last39_data["train_dataset"]),
            "valid_series": len(tft_last39_data["valid_dataset"]),
            "num_known_reals": tft_last39_data["num_known_reals"],
            "num_static_reals": tft_last39_data["num_static_reals"],
            "static_cat_cardinalities": str(tft_last39_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
        })

        result = fit_model(
            model=model,
            train_loader=tft_last39_data["train_loader"],
            valid_loader=tft_last39_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=tft_last39_data["holiday_feature_idx"],
            epochs=EPOCHS_LAST39_WEIGHTED,
            metric_prefix="last39",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("last39_best_valid_wmae", best_wmae)
        mlflow.log_metric("last39_best_epoch", best_epoch)

    tft_last39_weighted_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "trainable_parameters": n_params,
        "last39_best_valid_wmae": best_wmae,
        "last39_best_epoch": best_epoch,
    })

    print(f"Best last-39 weighted WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

tft_last39_weighted_results_df = (
    pd.DataFrame(tft_last39_weighted_results)
    .sort_values("last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_last39_weighted_results_df)

Last-39 weighted run 3/6: TFT_Last39WeightedHuber_h64_emb8_heads4_layers1_drop0.1_lr0.001_wd0.0001_hw5.0
{'embedding_dim': 8, 'hidden_dim': 64, 'lstm_layers': 1, 'attention_heads': 4, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'weighted_huber', 'holiday_weight': 5.0, 'delta': 1.0}
Trainable parameters: 125560
Epoch 001 | train_loss=0.31375 | valid_loss=0.43247 | valid_wmae=2648.79 | valid_mae=2584.60
Epoch 002 | train_loss=0.23910 | valid_loss=0.41870 | valid_wmae=2527.52 | valid_mae=2471.43
Epoch 003 | train_loss=0.20425 | valid_loss=0.41253 | valid_wmae=2436.22 | valid_mae=2403.96
Epoch 004 | train_loss=0.18641 | valid_loss=0.41188 | valid_wmae=2413.96 | valid_mae=2408.30
Epoch 005 | train_loss=0.17511 | valid_loss=0.40688 | valid_wmae=2402.04 | valid_mae=2393.25
Epoch 006 | train_loss=0.16702 | valid_loss=0.40041 | valid_wmae=2355.98 | valid_mae=2361.70
Epoch 007 | train_loss=0.16151 | valid_loss=0.40140 | valid_wmae=2366.64 | valid_mae=2368.77
Epoch 008 | tra

,run_name,run_id,embedding_dim,hidden_dim,lstm_layers,attention_heads,dropout,lr,weight_decay,loss_name,holiday_weight,delta,trainable_parameters,last39_best_valid_wmae,last39_best_epoch
0,TFT_Last39WeightedHuber_h128_emb8_heads4_layer...,5e30f2edf3124030b3b60d6eb3cec8c6,8,128,1,4,0.3,0.0005,0.0001,weighted_huber,5.0,1.0,479416,2245.839306,20
1,TFT_Last39WeightedHuber_h64_emb8_heads4_layers...,90e6241838c44b5f88592b3ff863b371,8,64,1,4,0.3,0.0005,0.0001,weighted_huber,5.0,1.0,125560,2249.678276,17
2,TFT_Last39WeightedHuber_h64_emb16_heads4_layer...,d4ec3e9075564ae5b75fa294b38cccf8,16,64,1,4,0.3,0.0010,0.0001,weighted_huber,5.0,1.0,128152,2271.150250,29
3,TFT_Last39WeightedHuber_h64_emb8_heads4_layers...,42b5d0c4b8ab4c669a1145fd07c8028b,8,64,1,4,0.1,0.0010,0.0001,weighted_huber,5.0,1.0,125560,2304.694567,23


In [ ]:
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    output_format="pandas",
)

last39_runs = runs[
    runs["tags.mlflow.runName"].astype(str).str.contains("TFT_Last39", na=False)
].copy()

cols = [
    "tags.mlflow.runName",
    "run_id",
    "params.hidden_dim",
    "params.embedding_dim",
    "params.attention_heads",
    "params.lstm_layers",
    "params.dropout",
    "params.learning_rate",
    "params.weight_decay",
    "params.loss_name",
    "params.holiday_weight",
    "metrics.last39_best_valid_wmae",
    "metrics.last39_best_epoch",
]

existing_cols = [c for c in cols if c in last39_runs.columns]

tft_last39_all_results_df = (
    last39_runs[existing_cols]
    .dropna(subset=["metrics.last39_best_valid_wmae"])
    .sort_values("metrics.last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(tft_last39_all_results_df)

,tags.mlflow.runName,run_id,params.hidden_dim,params.embedding_dim,params.attention_heads,params.lstm_layers,params.dropout,params.learning_rate,params.weight_decay,params.loss_name,params.holiday_weight,metrics.last39_best_valid_wmae,metrics.last39_best_epoch
0,TFT_Last39_h64_emb8_heads4_layers1_drop0.3_lr0...,ce7943e5789c4d0197f9da542266e435,64,8,4,1,0.3,0.001,0.0001,huber,None,2223.957789,13.0
1,TFT_Last39WeightedHuber_h128_emb8_heads4_layer...,5e30f2edf3124030b3b60d6eb3cec8c6,128,8,4,1,0.3,0.0005,0.0001,weighted_huber,5.0,2245.839306,20.0
2,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,38b0bc3c414c4fd2911f818f8af4d725,64,8,4,1,0.2,0.001,0.0001,huber,None,2248.005916,23.0
3,TFT_Last39_h64_emb8_heads4_layers1_drop0.1_lr0...,d8efc874e5f441e7983f50c7f2ca0735,64,8,4,1,0.1,0.001,0.0001,huber,None,2248.751290,15.0
4,TFT_Last39WeightedHuber_h64_emb8_heads4_layers...,90e6241838c44b5f88592b3ff863b371,64,8,4,1,0.3,0.0005,0.0001,weighted_huber,5.0,2249.678276,17.0
5,TFT_Last39_h64_emb8_heads4_layers1_drop0.2_lr0...,2bb0b16b545649e8917ef17b38cfdcd4,64,8,4,1,0.2,0.001,0.0005,huber,None,2253.931945,17.0
6,TFT_Last39_h64_emb16_heads4_layers1_drop0.2_lr...,4a018b71cf0c413cbd22c8267cb819e6,64,16,4,1,0.2,0.001,0.0001,huber,None,2259.420502,17.0
7,TFT_Last39_h128_emb8_heads4_layers1_drop0.2_lr...,11cadfcef12949958d419608f8ef56a2,128,8,4,1,0.2,0.001,0.0001,huber,None,2263.946137,8.0
8,TFT_Last39_h128_emb8_heads4_layers1_drop0.3_lr...,e08cf77ed7a24dc397b7d3d3dee3c0e2,128,8,4,1,0.3,0.0005,0.0001,huber,None,2266.685444,12.0
9,TFT_Last39WeightedHuber_h64_emb16_heads4_layer...,d4ec3e9075564ae5b75fa294b38cccf8,64,16,4,1,0.3,0.001,0.0001,weighted_huber,5.0,2271.150250,29.0


## Kaggle Check


In [37]:
# full train preprocessing for final test inference

final_base_preprocessor = WalmartBasePreprocessor()
final_base_preprocessor.fit(stores, features)

full_train_base = final_base_preprocessor.transform(train)

final_neural_preprocessor = WalmartNeuralPreprocessor()
final_neural_preprocessor.fit(full_train_base)

full_train_panel = final_neural_preprocessor.transform(full_train_base)

final_dataset_cols = final_neural_preprocessor.get_dataset_columns()

print("Full train panel shape:", full_train_panel.shape)
print("Full train date range:", full_train_panel["Date"].min(), "->", full_train_panel["Date"].max())
print("Full train unique dates:", full_train_panel["Date"].nunique())
print(final_dataset_cols)

Full train panel shape: (421570, 66)
Full train date range: 2010-02-05 00:00:00 -> 2012-10-26 00:00:00
Full train unique dates: 143
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': ['Store_id', 'Dept_id', 'Type_id'], 'static_real_cols': ['Size_scaled'], 'known_future_real_cols': ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']}


In [38]:
test_dates = pd.DataFrame({
    "Date": sorted(pd.to_datetime(test["Date"]).unique())
})

test_pairs = test[["Store", "Dept"]].drop_duplicates().reset_index(drop=True)

# cross join: every test pair gets all 39 test dates (later we will merge IsHoliday from original test )
test_grid = test_pairs.merge(test_dates, how="cross")

# IsHoliday depends on Date so map it from original test
date_holiday = (
    test[["Date", "IsHoliday"]]
    .copy()
    .assign(Date=lambda x: pd.to_datetime(x["Date"]))
    .drop_duplicates()
)

assert date_holiday["Date"].nunique() == len(date_holiday)

test_grid = test_grid.merge(date_holiday, on="Date", how="left")

print("Original test rows:", len(test))
print("Full test grid rows:", len(test_grid))
print("Test pairs:", len(test_pairs))
print("Test dates:", len(test_dates))

assert test_grid["IsHoliday"].notna().all()
assert len(test_dates) == PREDICTION_LENGTH

Original test rows: 115064
Full test grid rows: 123591
Test pairs: 3169
Test dates: 39


In [39]:
full_test_grid_base = final_base_preprocessor.transform(test_grid)
full_test_grid_panel = final_neural_preprocessor.transform(full_test_grid_base)

print("Full test grid panel shape:", full_test_grid_panel.shape)
print("Full test grid date range:", full_test_grid_panel["Date"].min(), "->", full_test_grid_panel["Date"].max())
print("Full test grid unique dates:", full_test_grid_panel["Date"].nunique())

test_group_sizes = full_test_grid_panel.groupby("series_id").size()
assert (test_group_sizes == PREDICTION_LENGTH).all()

print("All test Store-Dept groups have 39 rows.")

Full test grid panel shape: (123591, 65)
Full test grid date range: 2012-11-02 00:00:00 -> 2013-07-26 00:00:00
Full test grid unique dates: 39
All test Store-Dept groups have 39 rows.


In [40]:
BEST_TFT_FINAL_CONFIG_CALENDAR = {
    "embedding_dim": 8,
    "hidden_dim": 128,
    "lstm_layers": 1,
    "attention_heads": 4,
    "dropout": 0.3,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "loss_name": "weighted_huber",
    "holiday_weight": 5.0,
    "delta": 1.0,

    "final_epochs": 40,
}

# best result
BEST_TFT_FINAL_CONFIG_LAST39_HUBER = {
    "embedding_dim": 8,
    "hidden_dim": 64,
    "lstm_layers": 1,
    "attention_heads": 4,
    "dropout": 0.3,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",

    "final_epochs": 13,
}

BEST_TFT_FINAL_CONFIG_LAST39_WEIGHTED = {
    "embedding_dim": 8,
    "hidden_dim": 64,
    "lstm_layers": 1,
    "attention_heads": 4,
    "dropout": 0.3,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "weighted_huber",
    "holiday_weight": 5.0,
    "delta": 1.0,

    "final_epochs": 13,
}

BEST_TFT_FINAL_CONFIG_LAST39_WEIGHTED_2 = {
    "embedding_dim": 8,
    "hidden_dim": 128,
    "lstm_layers": 1,
    "attention_heads": 4,
    "dropout": 0.3,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "loss_name": "weighted_huber",
    "holiday_weight": 5.0,
    "delta": 1.0,
    
    "final_epochs": 20,
}

BEST_TFT_FINAL_CONFIG = BEST_TFT_FINAL_CONFIG_LAST39_HUBER

final_tft_known_future_cols = remove_markdown_features(
    final_dataset_cols["known_future_real_cols"]
)

final_tft_cols = {
    "target_col": final_dataset_cols["target_col"],
    "series_col": final_dataset_cols["series_col"],
    "static_cat_cols": final_dataset_cols["static_cat_cols"],
    "static_real_cols": final_dataset_cols["static_real_cols"],
    "known_future_real_cols": final_tft_known_future_cols,
}

FINAL_TFT_BATCH_SIZE = 512

final_tft_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    full_train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_tft_cols["target_col"],
    series_col=final_tft_cols["series_col"],
    static_cat_cols=final_tft_cols["static_cat_cols"],
    static_real_cols=final_tft_cols["static_real_cols"],
    known_future_real_cols=final_tft_cols["known_future_real_cols"],
)

final_tft_test_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=full_train_panel,
    future_df=full_test_grid_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_tft_cols["target_col"],
    series_col=final_tft_cols["series_col"],
    static_cat_cols=final_tft_cols["static_cat_cols"],
    static_real_cols=final_tft_cols["static_real_cols"],
    known_future_real_cols=final_tft_cols["known_future_real_cols"],
)

final_tft_train_loader = FastTensorDataLoader(
    final_tft_train_dataset.tensors,
    batch_size=FINAL_TFT_BATCH_SIZE,
    shuffle=True,
)

final_tft_test_loader = FastTensorDataLoader(
    final_tft_test_dataset.tensors,
    batch_size=FINAL_TFT_BATCH_SIZE,
    shuffle=False,
)

final_tft_static_cat_cardinalities = [
    int(max(full_train_panel[col].max(), full_test_grid_panel[col].max())) + 1
    for col in final_tft_cols["static_cat_cols"]
]

final_tft_num_static_reals = len(final_tft_cols["static_real_cols"])
final_tft_num_known_reals = len(final_tft_cols["known_future_real_cols"])
final_tft_holiday_feature_idx = final_tft_cols["known_future_real_cols"].index("IsHoliday")

print("Final TFT train windows:", len(final_tft_train_dataset))
print("Final TFT test forecast series:", len(final_tft_test_dataset))
print("Final TFT train batches:", len(final_tft_train_loader))
print("Final TFT test batches:", len(final_tft_test_loader))
print("Static cat cardinalities:", final_tft_static_cat_cardinalities)
print("Num known reals:", final_tft_num_known_reals)
print("Num static reals:", final_tft_num_static_reals)
print("Holiday idx:", final_tft_holiday_feature_idx)

assert final_tft_test_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
assert final_tft_test_dataset.tensors["future_known_reals"].shape[1] == PREDICTION_LENGTH

Final TFT train windows: 149069
Final TFT test forecast series: 3169
Final TFT train batches: 292
Final TFT test batches: 7
Static cat cardinalities: [46, 82, 4]
Num known reals: 11
Num static reals: 1
Holiday idx: 6


In [41]:
def make_final_tft_loss(config: dict, holiday_feature_idx: int):
    if config["loss_name"] == "weighted_huber":
        return WeightedHuberLoss(
            holiday_feature_idx=holiday_feature_idx,
            holiday_weight=config["holiday_weight"],
            delta=config["delta"],
        )

    return make_loss_fn(config["loss_name"])


def train_final_tft(
    config: dict,
    train_loader,
    device,
):
    model = TFTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=final_tft_static_cat_cardinalities,
        num_known_reals=final_tft_num_known_reals,
        num_static_reals=final_tft_num_static_reals,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        lstm_layers=config["lstm_layers"],
        attention_heads=config["attention_heads"],
        dropout=config["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_final_tft_loss(
        config=config,
        holiday_feature_idx=final_tft_holiday_feature_idx,
    )

    epochs = config["final_epochs"]

    print("Trainable parameters:", count_parameters(model))
    print("Final training epochs:", epochs)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
        )

        print(f"Epoch {epoch:03d} | final_train_loss={train_loss:.5f}")

    return model


@torch.no_grad()
def predict_tft(model, loader, device):
    model.eval()

    all_preds = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)

        preds_original = (
            preds_scaled * batch["target_std"].unsqueeze(1)
            + batch["target_mean"].unsqueeze(1)
        )

        all_preds.append(preds_original.detach().cpu().numpy())

    return np.concatenate(all_preds, axis=0)

In [42]:
set_seed(SEED + 900)

final_tft_model = train_final_tft(
    BEST_TFT_FINAL_CONFIG,
    final_tft_train_loader,
    DEVICE,
)

tft_test_preds_matrix = predict_tft(
    final_tft_model,
    final_tft_test_loader,
    DEVICE,
)

print("TFT test prediction matrix shape:", tft_test_preds_matrix.shape)
assert tft_test_preds_matrix.shape == (
    len(final_tft_test_dataset),
    PREDICTION_LENGTH,
)

Trainable parameters: 125560
Final training epochs: 13
Epoch 001 | final_train_loss=0.29494
Epoch 002 | final_train_loss=0.22520
Epoch 003 | final_train_loss=0.20251
Epoch 004 | final_train_loss=0.19148
Epoch 005 | final_train_loss=0.18365
Epoch 006 | final_train_loss=0.17727
Epoch 007 | final_train_loss=0.17108
Epoch 008 | final_train_loss=0.16606
Epoch 009 | final_train_loss=0.16144
Epoch 010 | final_train_loss=0.15737
Epoch 011 | final_train_loss=0.15445
Epoch 012 | final_train_loss=0.15158
Epoch 013 | final_train_loss=0.14913
TFT test prediction matrix shape: (3169, 39)


In [43]:
test_index_df = final_tft_test_dataset.get_future_index().reset_index(drop=True).copy()

tft_test_preds_flat = tft_test_preds_matrix.reshape(-1)

assert len(test_index_df) == len(tft_test_preds_flat)

full_tft_pred_df = test_index_df.copy()
full_tft_pred_df["Weekly_Sales"] = tft_test_preds_flat

negative_before_clip = (full_tft_pred_df["Weekly_Sales"] < 0).sum()
print("Negative predictions before clipping:", negative_before_clip)

# postprocessing: clip negative sales predictions to zero
full_tft_pred_df["Weekly_Sales"] = full_tft_pred_df["Weekly_Sales"].clip(lower=0)

print(full_tft_pred_df.head())
print(full_tft_pred_df["Weekly_Sales"].describe())

Negative predictions before clipping: 2372
   Store  Dept       Date  Weekly_Sales
0     10     1 2012-11-02  55932.789062
1     10     1 2012-11-09  43593.394531
2     10     1 2012-11-16  39062.160156
3     10     1 2012-11-23  38739.453125
4     10     1 2012-11-30  49000.785156
count    123591.000000
mean      15120.015625
std       22828.666016
min      -16703.503906
25%        1399.398010
50%        6506.809570
75%       19045.390625
max      672533.125000
Name: Weekly_Sales, dtype: float64


In [44]:
test_keys = test[["Store", "Dept", "Date"]].copy()
test_keys["Date"] = pd.to_datetime(test_keys["Date"])

tft_submission_df = test_keys.merge(
    full_tft_pred_df,
    on=["Store", "Dept", "Date"],
    how="left",
)

assert len(tft_submission_df) == len(test)
assert tft_submission_df["Weekly_Sales"].notna().all()

tft_submission_df["Id"] = (
    tft_submission_df["Store"].astype(str)
    + "_"
    + tft_submission_df["Dept"].astype(str)
    + "_"
    + tft_submission_df["Date"].dt.strftime("%Y-%m-%d")
)

tft_submission_df = tft_submission_df[["Id", "Weekly_Sales"]]

print(tft_submission_df.head())
print(tft_submission_df.shape)

submission_name = (
    "submission_tft_"
    f"h{BEST_TFT_FINAL_CONFIG['hidden_dim']}_"
    f"emb{BEST_TFT_FINAL_CONFIG['embedding_dim']}_"
    f"drop{BEST_TFT_FINAL_CONFIG['dropout']}_"
    f"{BEST_TFT_FINAL_CONFIG['loss_name']}_"
    f"epochs{BEST_TFT_FINAL_CONFIG['final_epochs']}.csv"
)

submission_path = repo_root / submission_name

tft_submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

               Id  Weekly_Sales
0  1_1_2012-11-02  29141.466797
1  1_1_2012-11-09  23050.925781
2  1_1_2012-11-16  21576.859375
3  1_1_2012-11-23  21160.853516
4  1_1_2012-11-30  25156.958984
(115064, 2)
Saved submission to: /kaggle/working/Walmart/submission_tft_h64_emb8_drop0.3_huber_epochs13.csv


There was a discrepancy between two validation split strategies: Last-39 split training ranked `h64_emb8_heads4_layers1_drop0.3_lr0.001_wd0.0001_huber` on top with best score. Calendar-aligned split ranked `h128_emb8_heads4_layers1_drop0.3_lr0.0005_wd0.0001_hw5.0` on top in respective group.

Interestingly, while Calendar-aligned split validation score seems to be the closest to the actual test set scores on Kaggle, model ranked higher by Last-39 split outperformed best Calendar-aligned model by 100. This could point to potential success in different blending strategies for those models.